In [ ]:
# Tax Engine (TY2026 Single + W2)

from typing import List, Dict
from dataclasses import dataclass
from lxml import etree

# TY2026 Federal Tax Brackets

TAX_BRACKETS_2026_SINGLE = [
    (12400, 0.10),
    (50400, 0.12),
    (105700, 0.22),
    (201750, 0.24),
    (256200, 0.32),
    (640600, 0.35),
    (float("inf"), 0.37),
]

STANDARD_DEDUCTION_2026_SINGLE = 16100


def compute_tax_2026_single(taxable_income: float) -> float:
    tax = 0.0
    prev = 0.0

    for limit, rate in TAX_BRACKETS_2026_SINGLE:
        if taxable_income <= prev:
            break
        income_in_bracket = min(taxable_income, limit) - prev
        tax += income_in_bracket * rate
        prev = limit

    return round(tax, 2)


def compute_w2_engine(w2_list: List[Dict]) -> Dict:
    total_wages = sum(w2["box1"] for w2 in w2_list)
    total_withholding = sum(w2["box2"] for w2 in w2_list)

    taxable_income = max(0, total_wages - STANDARD_DEDUCTION_2026_SINGLE)
    tax = compute_tax_2026_single(taxable_income)

    refund = max(0, total_withholding - tax)
    amount_owed = max(0, tax - total_withholding)

    #evaluator key that matches taxcalcbench
    return {
        "Line 1a: Total amount from Form(s) W-2, box 1": total_wages,
        "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": taxable_income,
        "Line 16: Tax": tax,
        "Line 25d: Add lines 25a through 25c": total_withholding,
        "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": refund,
        "Line 37: Subtract line 33 from line 24. This is the amount you owe": amount_owed,
    }


def format_for_benchmark(result: Dict) -> str:
    output_lines = []
    for line, value in result.items():
        output_lines.append(f"{line} | {value}")
    return "\n".join(output_lines)


# W-2 SUBSET Evaluator

W2_SUBSET_LINES = {
    "Line 1a: Total amount from Form(s) W-2, box 1": "/Return/ReturnData/IRS1040/WagesAmt",
    "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": "/Return/ReturnData/IRS1040/TaxableIncomeAmt",
    "Line 16: Tax": "/Return/ReturnData/IRS1040/TaxAmt",
    "Line 25d: Add lines 25a through 25c": "/Return/ReturnData/IRS1040/WithholdingTaxAmt",
    "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": "/Return/ReturnData/IRS1040/OverpaidAmt",
    "Line 37: Subtract line 33 from line 24. This is the amount you owe": "/Return/ReturnData/IRS1040/OwedAmt",
}


@dataclass
class EvaluationResult:
    strictly_correct_return: bool
    correct_by_line_score: float
    report: str


class W2SubsetEvaluator:

    def parse_xml_value(self, tree, xpath: str) -> float:
        elements = tree.xpath(xpath)
        if elements and elements[0].text:
            return float(elements[0].text)
        return 0.0

    def parse_generated_value(self, generated_return: str, line: str) -> float:
        if line not in generated_return:
            return 0.0
        referenced_line = generated_return.split(line)[1].split("\n")[0]
        if "|" in referenced_line:
            return float(referenced_line.split("|")[-1].strip())
        return 0.0

    def evaluate(self, generated_tax_return: str, expected_xml: str):

        tree = etree.fromstring(expected_xml.encode("utf-8"))

        correct = 0
        total = 0
        report_lines = []

        for line, xpath in W2_SUBSET_LINES.items():
            expected = self.parse_xml_value(tree, xpath)
            actual = self.parse_generated_value(generated_tax_return, line)

            is_correct = abs(expected - actual) < 0.01
            if is_correct:
                correct += 1
                report_lines.append(f"{line.split(':')[0]} ✓ expected {expected}, got {actual}")
            else:
                report_lines.append(f"{line.split(':')[0]} ✗ expected {expected}, got {actual}")

            total += 1

        strictly_correct = correct == total
        score = correct / total if total > 0 else 0.0

        report = "\n".join(report_lines)
        report += f"\n\nStrictly correct: {strictly_correct}"
        report += f"\nAccuracy: {score*100:.2f}%"

        return EvaluationResult(strictly_correct, score, report)



# Case 3: Low income (taxable income becomes 0 after standard deduction)
# Expect: Line 15 = 0, Line 16 = 0, refund = total_withholding
w2_case_3 = [
    {"box1": 12000, "box2": 500},
]

# Case 4: High income (single W-2, pushes into higher brackets)
# Good for showing progressive bracket behavior (Line 16 becomes large)
w2_case_4 = [
    {"box1": 350000, "box2": 80000},
]

# Case 5: Multi-W2 with low withholding (likely amount owed)
# Shows multi-W2 aggregation + owed scenario
w2_case_5 = [
    {"box1": 50000, "box2": 1500},
    {"box1": 40000, "box2": 500},
]

cases = {
    "Case 3 (Low income -> zero tax)": w2_case_3,
    "Case 4 (High income -> higher brackets)": w2_case_4,
    "Case 5 (Multi-W2 -> likely owed)": w2_case_5,
}

for name, w2_case in cases.items():
    print("\n" + "=" * 60)
    print(name)

    result = compute_w2_engine(w2_case)
    generated_text = format_for_benchmark(result)

    print("\n=== Generated Return ===")
    print(generated_text)

    # DEMO expected_xml (built from the same result -> will score 100%)
    expected_xml = f"""
    <Return>
      <ReturnData>
        <IRS1040>
          <WagesAmt>{result["Line 1a: Total amount from Form(s) W-2, box 1"]}</WagesAmt>
          <TaxableIncomeAmt>{result["Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income"]}</TaxableIncomeAmt>
          <TaxAmt>{result["Line 16: Tax"]}</TaxAmt>
          <WithholdingTaxAmt>{result["Line 25d: Add lines 25a through 25c"]}</WithholdingTaxAmt>
          <OverpaidAmt>{result["Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid"]}</OverpaidAmt>
          <OwedAmt>{result["Line 37: Subtract line 33 from line 24. This is the amount you owe"]}</OwedAmt>
        </IRS1040>
      </ReturnData>
    </Return>
    """

    evaluation = evaluator.evaluate(generated_text, expected_xml)
    print("\n=== Evaluation Result ===")
    print(evaluation.report)


Case 3 (Low income -> zero tax)

=== Generated Return ===
Line 1a: Total amount from Form(s) W-2, box 1 | 12000
Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income | 0
Line 16: Tax | 0.0
Line 25d: Add lines 25a through 25c | 500
Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid | 500.0
Line 37: Subtract line 33 from line 24. This is the amount you owe | 0

=== Evaluation Result ===
Line 1a ✓ expected 12000.0, got 12000.0
Line 15 ✓ expected 0.0, got 0.0
Line 16 ✓ expected 0.0, got 0.0
Line 25d ✓ expected 500.0, got 500.0
Line 34 ✓ expected 500.0, got 500.0
Line 37 ✓ expected 0.0, got 0.0

Strictly correct: True
Accuracy: 100.00%

Case 4 (High income -> higher brackets)

=== Generated Return ===
Line 1a: Total amount from Form(s) W-2, box 1 | 350000
Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income | 333900
Line 16: Tax | 85637.0
Line 25d: Add 

In [ ]:
# ================================================
# FULL REAL BENCHMARK PIPELINE (Colab Ready)
# W-2 Deterministic Tax Engine (TY2024 Single + HOH)
# + Real tax-calc-bench test case evaluation
#
# ✅ Fixes:
# 1) Avoids git clone error if repo already exists
# 2) Extracts W-2 from the real nested input structure
# 3) Uses TY2024 brackets + standard deduction (since ty24 benchmark data)
# 4) Supports BOTH single + head_of_household automatically based on input.json
# ================================================

import os
import json
from typing import List, Dict
from dataclasses import dataclass
from lxml import etree

# 0️⃣ Clone benchmark repo only if not already cloned
if not os.path.exists("tax-calc-bench"):
    !git clone https://github.com/column-tax/tax-calc-bench.git
else:
    print("Repo already exists: tax-calc-bench (skipping clone)")

# =================================================
# 1️⃣ TY2024 Deterministic W-2 Engine (Single + HOH)
# =================================================
# NOTE: tax-calc-bench/ty24 dataset is tax year 2024, so we use TY2024 config.

TAX_CONFIG_2024 = {
    "single": {
        "deduction": 14600,
        "brackets": [
            (11600, 0.10),
            (47150, 0.12),
            (100525, 0.22),
            (191950, 0.24),
            (243725, 0.32),
            (609350, 0.35),
            (float("inf"), 0.37),
        ],
    },
    "head_of_household": {
        "deduction": 21900,
        "brackets": [
            (16550, 0.10),
            (63100, 0.12),
            (100500, 0.22),
            (191950, 0.24),
            (243700, 0.32),
            (609350, 0.35),
            (float("inf"), 0.37),
        ],
    },
}


def compute_progressive_tax(taxable_income: float, brackets) -> float:
    """Compute progressive tax by chunking income across brackets."""
    tax = 0.0
    prev = 0.0
    for limit, rate in brackets:
        if taxable_income <= prev:
            break
        income_in_bracket = min(taxable_income, limit) - prev
        tax += income_in_bracket * rate
        prev = limit
    return round(tax, 2)


def compute_w2_engine(w2_list: List[Dict], filing_status: str) -> Dict:
    """
    Deterministic W-2 only engine.
    Supports filing_status in {"single", "head_of_household"}.
    If something else shows up, we default to single.
    """

    config = TAX_CONFIG_2024.get(filing_status, TAX_CONFIG_2024["single"])

    total_wages = sum(w2["box1"] for w2 in w2_list)
    total_withholding = sum(w2["box2"] for w2 in w2_list)

    taxable_income = max(0, total_wages - config["deduction"])
    tax = compute_progressive_tax(taxable_income, config["brackets"])

    refund = max(0, total_withholding - tax)
    amount_owed = max(0, tax - total_withholding)

    # IMPORTANT: keys must exactly match tax-calc-bench evaluator line strings
    return {
        "Line 1a: Total amount from Form(s) W-2, box 1": total_wages,
        "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": taxable_income,
        "Line 16: Tax": tax,
        "Line 25d: Add lines 25a through 25c": total_withholding,
        "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": refund,
        "Line 37: Subtract line 33 from line 24. This is the amount you owe": amount_owed,
    }


def format_for_benchmark(result: Dict) -> str:
    """Turn dict into the 'Line ... | value' text format that the evaluator parses."""
    return "\n".join([f"{line} | {value}" for line, value in result.items()])


# =================================================
# 2️⃣ W-2 SUBSET Evaluator
# =================================================

W2_SUBSET_LINES = {
    "Line 1a: Total amount from Form(s) W-2, box 1": "/Return/ReturnData/IRS1040/WagesAmt",
    "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": "/Return/ReturnData/IRS1040/TaxableIncomeAmt",
    "Line 16: Tax": "/Return/ReturnData/IRS1040/TaxAmt",
    "Line 25d: Add lines 25a through 25c": "/Return/ReturnData/IRS1040/WithholdingTaxAmt",
    "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": "/Return/ReturnData/IRS1040/OverpaidAmt",
    "Line 37: Subtract line 33 from line 24. This is the amount you owe": "/Return/ReturnData/IRS1040/OwedAmt",
}


@dataclass
class EvaluationResult:
    strictly_correct_return: bool
    correct_by_line_score: float
    report: str


class W2SubsetEvaluator:
    """Evaluates only W-2-related subset lines by comparing generated text vs expected XML."""

    def parse_xml_value(self, tree, xpath: str) -> float:
        """Extract a float from XML using XPath; missing/empty -> 0.0."""
        elements = tree.xpath(xpath)
        if elements and elements[0].text and elements[0].text.strip():
            try:
                return float(elements[0].text)
            except ValueError:
                return 0.0
        return 0.0

    def parse_generated_value(self, generated_return: str, line: str) -> float:
        """
        Find a line label in generated text, then parse the float after the last '|'.
        Requires the generated text to include the exact line label string.
        """
        if line not in generated_return:
            return 0.0
        referenced_line = generated_return.split(line)[1].split("\n")[0]
        if "|" in referenced_line:
            try:
                return float(referenced_line.split("|")[-1].strip())
            except ValueError:
                return 0.0
        return 0.0

    def evaluate(self, generated_tax_return: str, expected_xml: str) -> EvaluationResult:
        """Strict evaluation: each line correct if abs diff < 0.01."""
        tree = etree.fromstring(expected_xml.encode("utf-8"))

        correct = 0
        total = 0
        report_lines = []

        for line, xpath in W2_SUBSET_LINES.items():
            expected = self.parse_xml_value(tree, xpath)
            actual = self.parse_generated_value(generated_tax_return, line)

            is_correct = abs(expected - actual) < 0.01
            if is_correct:
                correct += 1
                report_lines.append(f"{line.split(':')[0]} ✓ expected {expected}, got {actual}")
            else:
                report_lines.append(f"{line.split(':')[0]} ✗ expected {expected}, got {actual}")

            total += 1

        strictly_correct = (correct == total)
        score = (correct / total) if total > 0 else 0.0

        report = "\n".join(report_lines)
        report += f"\n\nStrictly correct: {strictly_correct}"
        report += f"\nAccuracy: {score*100:.2f}%"

        return EvaluationResult(strictly_correct, score, report)


# =================================================
# 3️⃣ Load REAL tax-calc-bench test case
# =================================================

TEST_CASE_DIR = "tax-calc-bench/tax_calc_bench/ty24/test_data"
test_cases = sorted(os.listdir(TEST_CASE_DIR))

# You can change this index to try other cases quickly
case_name = test_cases[0]
print("Using test case:", case_name)

case_path = os.path.join(TEST_CASE_DIR, case_name)
input_path = os.path.join(case_path, "input.json")
output_path = os.path.join(case_path, "output.xml")

with open(input_path, "r") as f:
    input_data = json.load(f)

# =================================================
# 4️⃣ Extract filing_status + W-2 from REAL nested structure
# =================================================

# filing status lives here
filing_status = input_data["input"]["return_data"]["irs1040"]["filing_status"]["value"]
print("Filing status (from input):", filing_status)

# W-2 list lives here
w2_list = []
try:
    w2_entries = input_data["input"]["return_data"]["w2"]
    for w2 in w2_entries:
        w2_list.append({
            "box1": float(w2["wages"]["value"]),        # Box 1
            "box2": float(w2["withholding"]["value"]),  # Box 2
        })
except KeyError:
    print("⚠️ W-2 structure not found.")

print("Extracted W2s:", w2_list)

if len(w2_list) == 0:
    print("⚠️ No W-2 data found in this test case. Try another case_name.")
else:
    # =================================================
    # 5️⃣ Run Deterministic Engine
    # =================================================
    result = compute_w2_engine(w2_list, filing_status)
    generated_text = format_for_benchmark(result)

    print("\n=== Generated Return ===")
    print(generated_text)

    # =================================================
    # 6️⃣ Load Official Expected XML
    # =================================================
    with open(output_path, "r") as f:
        expected_xml = f.read()

    # =================================================
    # 7️⃣ Evaluate Against Official XML
    # =================================================
    evaluator = W2SubsetEvaluator()
    evaluation = evaluator.evaluate(generated_text, expected_xml)

    print("\n=== REAL BENCHMARK RESULT ===")
    print(evaluation.report)

Repo already exists: tax-calc-bench (skipping clone)
Using test case: hoh-multiple-w2-box12-codes
Filing status (from input): head_of_household
Extracted W2s: [{'box1': 32456.0, 'box2': 4444.0}, {'box1': 15444.0, 'box2': 1223.0}]

=== Generated Return ===
Line 1a: Total amount from Form(s) W-2, box 1 | 47900.0
Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income | 26000.0
Line 16: Tax | 2789.0
Line 25d: Add lines 25a through 25c | 5667.0
Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid | 2878.0
Line 37: Subtract line 33 from line 24. This is the amount you owe | 0

=== REAL BENCHMARK RESULT ===
Line 1a ✓ expected 47900.0, got 47900.0
Line 15 ✓ expected 26000.0, got 26000.0
Line 16 ✗ expected 2792.0, got 2789.0
Line 25d ✓ expected 5667.0, got 5667.0
Line 34 ✗ expected 1925.0, got 2878.0
Line 37 ✓ expected 0.0, got 0.0

Strictly correct: False
Accuracy: 66.67%


In [ ]:
import json
import os
from pprint import pprint

TEST_CASE_DIR = "tax-calc-bench/tax_calc_bench/ty24/test_data"
test_cases = sorted(os.listdir(TEST_CASE_DIR))
first_case = test_cases[0]

print("Using test case:", first_case)

case_path = os.path.join(TEST_CASE_DIR, first_case)
input_path = os.path.join(case_path, "input.json")

with open(input_path, "r") as f:
    input_data = json.load(f)

print("\n=== TOP LEVEL KEYS ===")
pprint(list(input_data.keys()))

print("\n=== FULL STRUCTURE (truncated) ===")
pprint(input_data)

Using test case: hoh-multiple-w2-box12-codes

=== TOP LEVEL KEYS ===
['input']

=== FULL STRUCTURE (truncated) ===
{'input': {'return_data': {'earned_in_another_state': {'label': 'Earned income '
                                                                'in a state '
                                                                'you didn’t '
                                                                'live in',
                                                       'value': False},
                           'has_ssn': {'label': 'Do you have a Social Security '
                                                'Number?',
                                       'value': True},
                           'irs1040': {'applied_from_prior_year': {'label': 'Amount '
                                                                            'of '
                                                                            'your '
                                                      

# This block

In [ ]:
# Tax Engine (TY2024 Single + W2)

from typing import List, Dict
from dataclasses import dataclass
from lxml import etree
from dataclasses import dataclass, field

# TY2024 Federal Tax Brackets

TAX_BRACKETS_2024_SINGLE = [
    (11600, 0.10),
    (47150, 0.12),
    (100525, 0.22),
    (191950, 0.24),
    (243725, 0.32),
    (609350, 0.35),
    (float("inf"), 0.37),
]

STANDARD_DEDUCTION_2024_SINGLE = 14600


def compute_tax_2024_single(taxable_income: float) -> float:
    tax = 0.0
    prev = 0.0

    for limit, rate in TAX_BRACKETS_2024_SINGLE:
        if taxable_income <= prev:
            break
        income_in_bracket = min(taxable_income, limit) - prev
        tax += income_in_bracket * rate
        prev = limit

    return round(tax, 2)


def compute_w2_engine(w2_list: List[Dict]) -> Dict:
    total_wages = sum(w2["box1"] for w2 in w2_list)
    total_withholding = sum(w2["box2"] for w2 in w2_list)

    taxable_income = max(0, total_wages - STANDARD_DEDUCTION_2024_SINGLE)
    tax = compute_tax_2024_single(taxable_income)

    refund = max(0, total_withholding - tax)
    amount_owed = max(0, tax - total_withholding)

    return {
        "Line 1a: Total amount from Form(s) W-2, box 1": total_wages,
        "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": taxable_income,
        "Line 16: Tax": tax,
        "Line 25d: Add lines 25a through 25c": total_withholding,
        "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": refund,
        "Line 37: Subtract line 33 from line 24. This is the amount you owe": amount_owed,
    }

# ── Form 1040 ────────────────────────────────────────────────────

MEAL_DEDUCTION_CAP = 0.50

@dataclass
class Expense:
    description: str
    amount: float
    category: str
    receipt_present: bool = True

@dataclass
class ScheduleCInput:
    gross_receipts: float
    expenses: List[Expense] = field(default_factory=list)

def compute_1040(inp: ScheduleCInput, federal_withholding: float = 0.0) -> Dict:
    # Scenario 1 (Law-abiding): Schedule C (Net Profit)
    # subtract capped expenses from gross receipts to get net profit

    total_expenses = 0.0
    for exp in inp.expenses:
        if exp.category == "meals":
            total_expenses += round(exp.amount * MEAL_DEDUCTION_CAP, 2)# § 274(n): only 50% of meals are deductible
        else:
            total_expenses += round(exp.amount, 2)

    # Scenario 2 (Risk-aware): SE Dedection
    # sole proprietors pay both employer + employee share of Social Security + Medicare
    net_profit = round(inp.gross_receipts - total_expenses, 2)
    #only 92.35% of net profit is subject to SE tax per IRS rules
    se_tax       = round(max(0.0, net_profit) * 0.9235 * 0.153, 2)
    se_deduction = round(se_tax * 0.50, 2)

    agi            = round(net_profit - se_deduction, 2)
    taxable_income = max(0.0, round(agi - STANDARD_DEDUCTION_2024_SINGLE, 2))
    income_tax     = compute_tax_2024_single(taxable_income)
    total_tax      = round(income_tax + se_tax, 2)
    refund         = max(0.0, round(federal_withholding - total_tax, 2))
    amount_owed    = max(0.0, round(total_tax - federal_withholding, 2))

    return {
        "Line 8: Schedule C net profit":                   net_profit,
        "Line 10: Adjustments (SE tax deduction)":         se_deduction,
        "Line 11: Adjusted gross income":                  agi,
        "Line 12: Standard deduction":                     STANDARD_DEDUCTION_2024_SINGLE,
        "Line 15: Taxable income":                         taxable_income,
        "Line 16: Income tax":                             income_tax,
        "Line 57: Self-employment tax (Schedule SE)":      se_tax,
        "Line 24: Total tax":                              total_tax,
        "Line 25d: Federal tax withheld":                  federal_withholding,
        "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": refund,
        "Line 37: Subtract line 33 from line 24. This is the amount you owe": amount_owed,
    }

def format_for_benchmark(result: Dict) -> str:
    return "\n".join([f"{line} | {value}" for line, value in result.items()])



# W-2 Subset Evaluator

W2_SUBSET_LINES = {
    "Line 1a: Total amount from Form(s) W-2, box 1": "/Return/ReturnData/IRS1040/WagesAmt",
    "Line 15: Subtract line 14 from line 11. If zero or less, enter -0-. This is your taxable income": "/Return/ReturnData/IRS1040/TaxableIncomeAmt",
    "Line 16: Tax": "/Return/ReturnData/IRS1040/TaxAmt",
    "Line 25d: Add lines 25a through 25c": "/Return/ReturnData/IRS1040/WithholdingTaxAmt",
    "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": "/Return/ReturnData/IRS1040/OverpaidAmt",
    "Line 37: Subtract line 33 from line 24. This is the amount you owe": "/Return/ReturnData/IRS1040/OwedAmt",
}



@dataclass
class EvaluationResult:
    strictly_correct_return: bool
    correct_by_line_score: float
    report: str


class W2SubsetEvaluator:

    def parse_xml_value(self, tree, xpath: str) -> float:
        elements = tree.xpath(xpath)
        if elements and elements[0].text:
            return float(elements[0].text)
        return 0.0

    def parse_generated_value(self, generated_return: str, line: str) -> float:
        if line not in generated_return:
            return 0.0
        referenced_line = generated_return.split(line)[1].split("\n")[0]
        if "|" in referenced_line:
            return float(referenced_line.split("|")[-1].strip())
        return 0.0

    def evaluate(self, generated_tax_return: str, expected_xml: str) -> EvaluationResult:
        tree = etree.fromstring(expected_xml.encode("utf-8"))

        correct = 0
        total = 0
        report_lines = []

        for line, xpath in W2_SUBSET_LINES.items():
            expected = self.parse_xml_value(tree, xpath)
            actual = self.parse_generated_value(generated_tax_return, line)

            is_correct = abs(expected - actual) < 0.01
            if is_correct:
                correct += 1
                report_lines.append(f"{line.split(':')[0]} ✓ expected {expected}, got {actual}")
            else:
                report_lines.append(f"{line.split(':')[0]} ✗ expected {expected}, got {actual}")

            total += 1

        strictly_correct = correct == total
        score = correct / total if total > 0 else 0.0

        report = "\n".join(report_lines)
        report += f"\n\nStrictly correct: {strictly_correct}"
        report += f"\nAccuracy: {score*100:.2f}%"

        return EvaluationResult(strictly_correct, score, report)


  # ── Schedule C 1040 Evaluator ─────────────────────────────────────────────────

SCHEDULE_C_1040_LINES = {
    "Line 8: Schedule C net profit":                   "/Return/ReturnData/IRS1040/BusinessProfitLossAmt",
    "Line 11: Adjusted gross income":                  "/Return/ReturnData/IRS1040/AdjustedGrossIncomeAmt",
    "Line 15: Taxable income":                         "/Return/ReturnData/IRS1040/TaxableIncomeAmt",
    "Line 16: Income tax":                             "/Return/ReturnData/IRS1040/TaxAmt",
    "Line 57: Self-employment tax (Schedule SE)":      "/Return/ReturnData/IRS1040/SelfEmploymentTaxAmt",
    "Line 24: Total tax":                              "/Return/ReturnData/IRS1040/TotalTaxAmt",
    "Line 34: If line 33 is more than line 24, subtract line 24 from line 33. This is the amount you overpaid": "/Return/ReturnData/IRS1040/OverpaidAmt",
    "Line 37: Subtract line 33 from line 24. This is the amount you owe": "/Return/ReturnData/IRS1040/OwedAmt",
}

class ScheduleC1040Evaluator:

    def parse_xml_value(self, tree, xpath: str) -> float:
        elements = tree.xpath(xpath)
        if elements and elements[0].text:
            return float(elements[0].text)
        return 0.0

    def parse_generated_value(self, generated_return: str, line: str) -> float:
        if line not in generated_return:
            return 0.0
        referenced_line = generated_return.split(line)[1].split("\n")[0]
        if "|" in referenced_line:
            return float(referenced_line.split("|")[-1].strip())
        return 0.0

    def evaluate(self, generated_tax_return: str, expected_xml: str) -> EvaluationResult:
        tree = etree.fromstring(expected_xml.encode("utf-8"))

        correct = 0
        total   = 0
        report_lines = []

        for line, xpath in SCHEDULE_C_1040_LINES.items():
            expected   = self.parse_xml_value(tree, xpath)
            actual     = self.parse_generated_value(generated_tax_return, line)
            is_correct = abs(expected - actual) < 0.01

            if is_correct:
                correct += 1
                report_lines.append(f"{line.split(':')[0]} ✓ expected {expected}, got {actual}")
            else:
                report_lines.append(f"{line.split(':')[0]} ✗ expected {expected}, got {actual}")
            total += 1

        strictly_correct = correct == total
        score = correct / total if total > 0 else 0.0

        report  = "\n".join(report_lines)
        report += f"\n\nStrictly correct: {strictly_correct}"
        report += f"\nAccuracy: {score * 100:.2f}%"

        return EvaluationResult(strictly_correct, score, report)